# Day 0 - environment test

Run this at the start of each session. It checks imports, runs two squidpy functions on a synthetic object, and confirms the shared data is reachable. No internet needed.

### Setup (run first)

On the course servers this does nothing, since the packages are pre-installed. On Google Colab or a fresh laptop it installs what is missing. Data downloads automatically the first time it is needed, except the Xenium bundle on Day 2 (see the README), which can be manually downloaded or use the version available at /home/shared/spatial_course_data/xenium_kidney.

In [2]:
import importlib, subprocess, sys
_req = [('scanpy','scanpy'), ('squidpy','squidpy'), ('leidenalg','leidenalg'),
        ('igraph','igraph'), ('pyclustree','pyclustree'),
        ('scikit-image','skimage'), ('imageio','imageio')]
_missing = [pip for pip, mod in _req if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)
print('environment ready')

environment ready


In [3]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.titlesize": 11,
    "font.weight": "normal",
    "figure.dpi": 110,
})

def _data_dir():
    env = os.environ.get("SPATIAL_COURSE_DATA")
    if env:
        return Path(env)
    shared = Path("/home/shared/spatial_course_data")
    if shared.exists():
        return shared
    local = Path.home() / "spatial_course_data"
    local.mkdir(parents=True, exist_ok=True)
    return local

DATA_DIR = _data_dir()
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

def load_visium():
    p = DATA_DIR / "visium_lymph_node.h5ad"
    if p.exists():
        adata = sc.read_h5ad(p)
    else:
        adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
        try:
            adata.write(p)
        except Exception:
            pass
    adata.var_names_make_unique()
    return adata

def load_seqfish():
    p = DATA_DIR / "seqfish_embryo.h5ad"
    if p.exists():
        return sc.read_h5ad(p)
    adata = sq.datasets.seqfish()
    try:
        adata.write(p)
    except Exception:
        pass
    return adata

def xenium_dir():
    d = DATA_DIR / "xenium_kidney"
    if (d / "cell_feature_matrix.h5").exists():
        return d
    raise FileNotFoundError(
        f"Xenium bundle not found in {d}. See the README (Data): download the 10x "
        "Xenium Output Bundle once and unzip it there, or run scripts/prefetch_data.py."
    )

print("data dir:", DATA_DIR)

data dir: /home/shared/spatial_course_data


## Package versions

In [4]:
import anndata, sklearn
for name, mod in [('scanpy', sc), ('squidpy', sq), ('anndata', anndata), ('sklearn', sklearn)]:
    print(name, mod.__version__)

scanpy 1.11.5
squidpy 1.8.1
anndata 0.12.16
sklearn 1.9.0


/tmp/ipykernel_4630/3149343810.py:3: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(name, mod.__version__)
/tmp/ipykernel_4630/3149343810.py:3: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(name, mod.__version__)


## Tiny synthetic run

In [5]:
rng = np.random.default_rng(0)
n = 300
a = sc.AnnData(rng.poisson(1.0, size=(n, 20)).astype('float32'))
a.obs_names = [f'cell_{i}' for i in range(n)]
a.var_names = [f'gene_{j}' for j in range(20)]
a.obsm['spatial'] = rng.uniform(0, 100, size=(n, 2))
a.obs['group'] = pd.Categorical(rng.integers(0, 3, size=n).astype(str))
sq.gr.spatial_neighbors(a, coord_type='generic', n_neighs=6)
sq.gr.nhood_enrichment(a, cluster_key='group')
print('squidpy ok')

INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        


  0%|          | 0/1000 [00:00<?, ?/s]

squidpy ok


## Shared data check

In [6]:
expected = [
    'visium_lymph_node.h5ad',
    'xenium_kidney/cell_feature_matrix.h5',
    'seqfish_embryo.h5ad',
]
for rel in expected:
    p = DATA_DIR / rel
    print('OK  ' if p.exists() else 'MISS', rel)

MISS visium_lymph_node.h5ad
OK   xenium_kidney/cell_feature_matrix.h5
MISS seqfish_embryo.h5ad


If something is missing, let instructor know.